# Preprocessing
Prepare both corpora using shared stopwords.


## 1. Load data
Read the cleaned CSVs. Run from the notebook folder.


In [1]:
from pathlib import Path
import html
import re
import jieba
import pandas as pd

project = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebook").is_dir() and (p / "data").is_dir())
DATA = project / "data"

corpora = {
    "official": pd.read_csv(DATA / "official media cleaned.csv", dtype=str).fillna(""),
    "weibo": pd.read_csv(DATA / "weibo posts cleaned.csv", dtype=str).fillna("")
}

# Reviewed ads, institutional reposts, and unrelated posts.
EXCLUDED_WEIBO_ROWS = set("""
90 125 562 672 1207 1671 1979 2053 2256 2974 3168 5919 6611 6612 7050 7228 7244 7665 7843 8413 8556
8581 9169 10298 10422 10569 11139 12263 12271 13028 13030 13261 14233 14936 16247 16341 16343 16359
16522 16669 16949 17129 17487 17510 21632 22855 23208 23676 23942 23958 24513 24869 25101 25589
26195 27531 27665 27794 27819 27914 28373 28374 28701 28851 29095 30187 30882 30929 30932 30936
30938 31531 32200 32420 32956 33130 33133 33205 35557 35885 35893 36667 36693 38430
""".split())
corpora["weibo"] = corpora["weibo"][~corpora["weibo"]["source_row"].isin(EXCLUDED_WEIBO_ROWS)].copy()


## 2. Define rules
Set phrases and stopwords for both corpora.


In [2]:
PHRASES = set("""
不婚 不婚主义 不幸福 不想恋爱 不想相亲 不想结婚 不愿恋爱 不愿意结婚 不愿结婚 不敢恋爱 不敢结婚 不结婚 人身安全保护令 催婚催育 全国通办 初婚年龄 天价彩礼
夫妻共同债务 夫妻共同财产 婚俗改革 婚姻家庭纠纷 婚姻登记 婚恋交友 婚恋平台 婚恋观 家庭暴力 恋爱观 恐婚恐育 择偶标准 离婚冷静期 离婚登记 移风易俗 结婚意愿
结婚登记 跨省通办 零彩礼 高价彩礼
""".split())

NEGATIONS = set("""
不 不会 不必 不想 不愿 不敢 不是 不曾 不能 不要 从不 从未 别 勿 否 并不 并非 无 无法 未 未曾 没 没有 没法 莫 非
""".split())

FILLERS = set("""
一个 两个 两个人 事情 其实 就是 然后 真的 这样 这种
""".split())

PLATFORM_WORDS = set("""
全文 关注 博主 博文 图片 展开全文 微博 微博视频 点赞 热搜 私信 秒拍视频 粉丝 网友 网页链接 视频 评论 评论区 话题 超话 转发 转发微博 链接 长图
""".split())

NEWS_WORDS = set("""
上午 下午 下载客户端 中午 中国新闻网 中国日报 中国青年网 中央电视台 中新社 中新网 了解到 二维码 人民日报 人民网 今年 今日 介绍 作者 光明日报 光明网
原标题 去年 发布 央视 央视网 媒体报道 官网 客户端 强调 当天 微信公众号 快讯 扫一扫 投稿 投稿邮箱 报道 指出 据了解 摄制 摄影 新华社 新华网
新闻报道 新闻网 新闻记者 日前 日电 早上 昨日 晚间 未经授权 本报 本报讯 本报记者 本网 来源 此次 注册 消息 点击查看 点击进入 版权 版权声明 版权所有
现场 电话 目前 相关 禁止转载 稿件 编辑 网站 获悉 表示 认为 记者 责任编辑 责编 转载 转载请注明 近年来 近日 返回首页 进一步 进行 透露 通讯员 通讯社
通过 阅读原文
""".split())

MONEY_UNITS = set("万元 千元 百万元 亿元 万块 块钱 人民币 美元 日元 港元".split())

KEEP_SINGLE = set("""
娶 婚 嫁
""".split())

STOPWORD_FILES = ["hit_stopwords.txt", "baidu_stopwords.txt", "scu_stopwords.txt", "cn_stopwords.txt"]
public_stopwords = set()
for filename in STOPWORD_FILES:
    path = DATA / "stopwords" / filename
    public_stopwords.update(path.read_text(encoding="utf-8-sig").lower().split())

stopwords = ((public_stopwords | FILLERS | PLATFORM_WORDS | NEWS_WORDS | MONEY_UNITS)
             - PHRASES - KEEP_SINGLE) | NEGATIONS

tokenizer = jieba.Tokenizer()
for word in sorted(PHRASES | FILLERS | PLATFORM_WORDS | NEWS_WORDS | NEGATIONS):
    tokenizer.add_word(word, freq=100000)


## 3. Clean formatting
Remove links and markup; keep hashtag text.


In [3]:
URL = re.compile(r"(?:https?://|www\.|(?:[A-Za-z0-9-]+\.)+(?:com|cn|net|org|edu|gov|io)/)[A-Za-z0-9._~:/?#\[\]@!$&'()*+,;=%+\-]+", re.I)
EMAIL = re.compile(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}")
HTML = re.compile(r"</?[A-Za-z][A-Za-z0-9]*(?:\s+[^<>]*?)?\s*/?>")
ERROR_VALUES = {"#NAME?", "#VALUE!", "#REF!", "#DIV/0!", "#N/A", "#NUM!", "#NULL!"}

def clean_formatting(text):
    if text.strip().upper() in ERROR_VALUES:
        raise ValueError("Corrupt source text; check the cleaned CSV.")
    text = html.unescape(text)
    text = re.sub(r"（本条新闻\s*\|.*$", "", text, flags=re.S)
    text = text.replace("发布了头条文章", " ")
    text = URL.sub(" ", text)
    text = EMAIL.sub(" ", text)
    text = HTML.sub(" ", text)
    text = re.sub(r"#([^#]+)#", r" \1 ", text)
    text = re.sub(r"(?:O网页链接|网页链接|展开全文|收起全文|微博视频|秒拍视频)", " ", text)
    text = re.sub(r"[\u200b-\u200f\u202a-\u202e\u2060-\u206f\ufeff\x00]", "", text)
    return re.sub(r"\s+", " ", text).strip()


## 4. Tokenize
Split text into words and remove stopwords.


In [4]:
VALID_WORD = re.compile(r"[\u4e00-\u9fffA-Za-z0-9.%]+")
NUMBER = re.compile(r"\d+(?:\.\d+)?(?:%|[wk])?")
FORMAT_TOKEN = re.compile(r"p\d+|jpe?g|png|gif|webp|h{2,}|em{2,}")
REPEATED = re.compile(r"([\u4e00-\u9fff])\1{2,}")

def tokenize(text):
    words = []
    for word in tokenizer.lcut(text):
        word = word.strip().lower()
        if not word or word in stopwords or FORMAT_TOKEN.fullmatch(word):
            continue
        if not VALID_WORD.fullmatch(word) or NUMBER.fullmatch(word) or REPEATED.fullmatch(word):
            continue
        if len(word) == 1 and word not in KEEP_SINGLE:
            continue
        words.append(word)
    return words


## 5. Process texts
Use title + body for official articles and body for Weibo.


In [5]:
processed = {}
for name, frame in corpora.items():
    result = frame.drop(columns=["tokens_basic", "token_count_basic"], errors="ignore").copy()
    texts = frame["text"]
    if name == "official":
        texts = frame["title"].str.strip() + "。" + texts
    result["analysis_text"] = texts.map(clean_formatting)
    tokens = result["analysis_text"].map(tokenize)
    result["tokens"] = tokens.map(" ".join)
    result["token_count"] = tokens.map(len)
    processed[name] = result


## 6. Export
Save analysis CSVs, excluding texts with no remaining tokens.


In [6]:
OUTPUTS = {
    "official": DATA / "official_media_analysis.csv",
    "weibo": DATA / "weibo_posts_analysis.csv"
}

for name, frame in processed.items():
    output = frame[frame["token_count"] > 0]
    output.to_csv(OUTPUTS[name], index=False, encoding="utf-8-sig")
    print(f"{name}: saved {len(output):,} documents")


official: saved 2,039 documents
weibo: saved 22,530 documents
